# Text-to-Text (TTT) — local LLM inference on Apple Silicon

Pick a **model family** (Phi-3, Llama-3, Falcon3, Qwen2.5, Llama-2), a **size** (SMALL / MEDIUM / LARGE) and,
when a size holds more than one model, its **parameter count** (e.g. Qwen2.5 SMALL: 0.5B, 1.5B or 3B),
then run the pipeline end to end:

1. Select the model from the `MODEL_NAME` registry (`APPLE → TTT → FAMILY → SIZE → PARAMS`)
2. Review every family and size: parameters, memory footprint and cost per input token
3. Load the model
4. Load a prompt set (`prompts/tti_tts_ttt_prompts/<set>.csv`) and count input tokens
5. Generate a response to every prompt
6. Save the responses
7. Measure the real cost of the run, per prompt and in total
8. Compare every model of the selected family that fits on this Mac, side by side

All models run on **MLX** (`mlx-lm`) on the Apple GPU as **4-bit** quantized, instruction-tuned checkpoints,
so every family and size is compared like for like.

## 1. Configuration — model registry

In [1]:
import os
from pathlib import Path
import sys

# Run from the project root, whether this notebook is opened from notebooks/ or from the root
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'prompts').is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# APPLE -> TTT -> family -> size -> parameter count -> model
# Sizes are relative to each family's full line-up: SMALL = its smallest models, LARGE = its biggest.
# The parameter count is also the output subfolder name: outputs/ttt/<family>/<size>/<params>/
MODEL_NAME = {
    'APPLE': {
        'TTT': {
            'Phi-3': {
                'SMALL':  {'3.8B': 'Phi-3-mini-128k-instruct'},     #  3.8B
                'MEDIUM': {'7B':   'Phi-3-small-128k-instruct'},    #  7.4B
                'LARGE':  {'14B':  'Phi-3-medium-128k-instruct'},   # 14.0B
            },
            'Llama-3': {
                'SMALL':  {'1B':  'Llama-3.2-1B-Instruct',          #  1.2B
                           '3B':  'Llama-3.2-3B-Instruct'},         #  3.2B
                'MEDIUM': {'8B':  'Llama-3.1-8B-Instruct'},         #  8.0B
                'LARGE':  {'70B': 'Llama-3.3-70B-Instruct'},        # 70.6B, needs ~40 GB: too big for a 48 GB Mac
                # Llama-3.1-405B (~228 GB at 4-bit) has no MLX 4-bit upload and fits no Mac, so it isn't listed
            },
            'Falcon3': {
                'SMALL':  {'1B':  'Falcon3-1B-Instruct',            #  1.7B
                           '3B':  'Falcon3-3B-Instruct'},           #  3.2B
                'MEDIUM': {'7B':  'Falcon3-7B-Instruct'},           #  7.5B
                'LARGE':  {'10B': 'Falcon3-10B-Instruct'},          # 10.3B
            },
            'Qwen2.5': {
                'SMALL':  {'0.5B': 'Qwen2.5-0.5B-Instruct',         #  0.5B
                           '1.5B': 'Qwen2.5-1.5B-Instruct',         #  1.5B
                           '3B':   'Qwen2.5-3B-Instruct'},          #  3.1B
                'MEDIUM': {'7B':   'Qwen2.5-7B-Instruct',           #  7.6B
                           '14B':  'Qwen2.5-14B-Instruct'},         # 14.8B
                'LARGE':  {'32B':  'Qwen2.5-32B-Instruct',          # 32.8B
                           '72B':  'Qwen2.5-72B-Instruct'},         # 72.7B, needs ~41 GB: too big for a 48 GB Mac
            },
            'Llama-2': {                          # base (not chat) models
                'SMALL':  {'7B':  'Llama-2-7b-hf'},                 #  6.7B
                'MEDIUM': {'13B': 'Llama-2-13b-hf'},                # 13.0B
                'LARGE':  {'70B': 'Llama-2-70b-hf'},                # 69.0B, needs ~39 GB: too big for a 48 GB Mac
            },
        },
    },
}

# Every model -> (family, size, parameter folder)
MODELS = {name: (family, size, params)
          for family, sizes in MODEL_NAME['APPLE']['TTT'].items()
          for size, by_params in sizes.items()
          for params, name in by_params.items()}

# Model -> 4-bit MLX checkpoint to run, and the original checkpoint it was made from
MLX_REPOS = {
    'Phi-3-mini-128k-instruct':   ('mlx-community/Phi-3-mini-128k-instruct-4bit',   'microsoft/Phi-3-mini-128k-instruct'),
    # The two Phi-3 models below are built in models/ on first use (see LOCAL_BUILDS and section 3)
    'Phi-3-small-128k-instruct':  ('models/Phi-3-small-128k-instruct-4bit',         'microsoft/Phi-3-small-128k-instruct'),
    'Phi-3-medium-128k-instruct': ('models/Phi-3-medium-128k-instruct-4bit',        'microsoft/Phi-3-medium-128k-instruct'),
    'Llama-3.2-1B-Instruct':      ('mlx-community/Llama-3.2-1B-Instruct-4bit',      'meta-llama/Llama-3.2-1B-Instruct'),
    'Llama-3.2-3B-Instruct':      ('mlx-community/Llama-3.2-3B-Instruct-4bit',      'meta-llama/Llama-3.2-3B-Instruct'),
    'Llama-3.1-8B-Instruct':      ('mlx-community/Meta-Llama-3.1-8B-Instruct-4bit', 'meta-llama/Llama-3.1-8B-Instruct'),
    'Llama-3.3-70B-Instruct':     ('mlx-community/Llama-3.3-70B-Instruct-4bit',     'meta-llama/Llama-3.3-70B-Instruct'),
    'Falcon3-1B-Instruct':        ('mlx-community/Falcon3-1B-Instruct-4bit',        'tiiuae/Falcon3-1B-Instruct'),
    'Falcon3-3B-Instruct':        ('mlx-community/Falcon3-3B-Instruct-4bit',        'tiiuae/Falcon3-3B-Instruct'),
    'Falcon3-7B-Instruct':        ('mlx-community/Falcon3-7B-Instruct-4bit',        'tiiuae/Falcon3-7B-Instruct'),
    'Falcon3-10B-Instruct':       ('mlx-community/Falcon3-10B-Instruct-4bit',       'tiiuae/Falcon3-10B-Instruct'),
    'Qwen2.5-0.5B-Instruct':      ('mlx-community/Qwen2.5-0.5B-Instruct-4bit',      'Qwen/Qwen2.5-0.5B-Instruct'),
    'Qwen2.5-1.5B-Instruct':      ('mlx-community/Qwen2.5-1.5B-Instruct-4bit',      'Qwen/Qwen2.5-1.5B-Instruct'),
    'Qwen2.5-3B-Instruct':        ('mlx-community/Qwen2.5-3B-Instruct-4bit',        'Qwen/Qwen2.5-3B-Instruct'),
    'Qwen2.5-7B-Instruct':        ('mlx-community/Qwen2.5-7B-Instruct-4bit',        'Qwen/Qwen2.5-7B-Instruct'),
    'Qwen2.5-14B-Instruct':       ('mlx-community/Qwen2.5-14B-Instruct-4bit',       'Qwen/Qwen2.5-14B-Instruct'),
    'Qwen2.5-32B-Instruct':       ('mlx-community/Qwen2.5-32B-Instruct-4bit',       'Qwen/Qwen2.5-32B-Instruct'),
    'Qwen2.5-72B-Instruct':       ('mlx-community/Qwen2.5-72B-Instruct-4bit',       'Qwen/Qwen2.5-72B-Instruct'),
    # Llama-2 has no MLX 4-bit uploads: converted locally on first use from ungated copies of the
    # meta-llama originals (identical weights)
    'Llama-2-7b-hf':              ('models/Llama-2-7b-hf-4bit',                     'NousResearch/Llama-2-7b-hf'),
    'Llama-2-13b-hf':             ('models/Llama-2-13b-hf-4bit',                    'NousResearch/Llama-2-13b-hf'),
    'Llama-2-70b-hf':             ('models/Llama-2-70b-hf-4bit',                    'NousResearch/Llama-2-70b-hf'),
}

# Models prepared locally in models/ the first time they are selected:
#   'convert':  no working MLX upload exists, so quantize the original checkpoint to 4-bit MLX
#   'fix_rope': the MLX upload's config uses the legacy rope type 'su', which transformers 5 rejects;
#               link its weights (no copy) and rename the type to 'longrope' (the same method)
LOCAL_BUILDS = {
    'Phi-3-small-128k-instruct':  ('convert', 'microsoft/Phi-3-small-128k-instruct'),
    'Phi-3-medium-128k-instruct': ('fix_rope', 'mlx-community/Phi-3-medium-128k-instruct-4bit'),
    'Llama-2-7b-hf':              ('convert', 'NousResearch/Llama-2-7b-hf'),
    'Llama-2-13b-hf':             ('convert', 'NousResearch/Llama-2-13b-hf'),
    'Llama-2-70b-hf':             ('convert', 'NousResearch/Llama-2-70b-hf'),
}

# Phi-3 ends each chat turn with <|end|>, but these MLX uploads don't list it as a stop token,
# so without this the model keeps writing new turns until MAX_NEW_TOKENS
EXTRA_STOP_TOKENS = {
    'Phi-3-mini-128k-instruct': ['<|end|>'],
    'Phi-3-medium-128k-instruct': ['<|end|>'],
}

# Phi-3-small ships its own tokenizer code (tiktoken-based), which must be allowed to run
TRUST_REMOTE_CODE = {'Phi-3-small-128k-instruct'}

PLATFORM, TASK = 'APPLE', 'TTT'
# Environment variables TTT_FAMILY / TTT_SIZE / TTT_PARAMS override these, for running from the terminal
FAMILY = os.environ.get('TTT_FAMILY', 'Llama-3')   # 'Phi-3' | 'Llama-3' | 'Falcon3' | 'Qwen2.5' | 'Llama-2'
SIZE = os.environ.get('TTT_SIZE', 'MEDIUM')         # 'SMALL' | 'MEDIUM' | 'LARGE'
# Parameter count within SIZE (e.g. '1.5B'); left empty, the smallest model of that size is used
PARAMS = os.environ.get('TTT_PARAMS') or next(iter(MODEL_NAME[PLATFORM][TASK][FAMILY][SIZE]))

model_name = MODEL_NAME[PLATFORM][TASK][FAMILY][SIZE][PARAMS]
repo, source_repo = MLX_REPOS[model_name]

# Prompt set: a CSV with columns id,prompt in prompts/tti_tts_ttt_prompts/. TTT_PROMPTS overrides it from the terminal.
PROMPT_SET = os.environ.get('TTT_PROMPTS', 'ttt_prompts_500w')   # a CSV in prompts/tti_tts_ttt_prompts/ with columns id,prompt
INPUT_PATH = Path('prompts/tti_tts_ttt_prompts') / f'{PROMPT_SET}.csv'
OUTPUT_DIR = Path('outputs/ttt') / FAMILY   # outputs/ttt/<family>/<size>/<params>/<prompt set>/ (see out_dir, section 2)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_NEW_TOKENS = min(int(os.environ.get('TTT_MAX_TOKENS', '1024')), 1024)   # hard cap: no response exceeds 1024 output tokens
POWER_SAMPLE_MS = 100            # macmon sampling interval (its fastest is ~110 ms)
ELECTRICITY_USD_PER_KWH = 0.15   # your electricity price, for the $ per 1M tokens figures
RUN_FAMILY_COMPARISON = os.environ.get('TTT_COMPARE', '1') == '1'   # section 8: also run the other models of FAMILY

print(f'{PLATFORM} / {TASK} / {FAMILY} / {SIZE} / {PARAMS} -> {model_name} -> {repo}')
print(f'Prompt set: {INPUT_PATH}  |  max new tokens: {MAX_NEW_TOKENS}')

APPLE / TTT / Llama-3 / MEDIUM / 8B -> Llama-3.1-8B-Instruct -> mlx-community/Meta-Llama-3.1-8B-Instruct-4bit
Prompt set: prompts/tti_tts_ttt_prompts/ttt_prompts_500w.csv  |  max new tokens: 1024


## 2. Model families — sizes, memory and input token cost

Architecture numbers come from each checkpoint's `config.json`; parameter counts from the original
checkpoint's safetensors metadata on Hugging Face.

| Column | Formula |
|---|---|
| Weights bf16 / 8-bit / 4-bit | params × 2 bytes / × 8.5 bits / × 4.5 bits (MLX stores a scale and bias per 64 weights, adding 0.5 bits) |
| 4-bit download | actual size of the MLX checkpoint this notebook runs |
| KV cache per input token | 2 (K and V) × layers × KV heads × head dim × 2 bytes |
| Memory at full context | 4-bit weights + KV cache × context window |
| Max context on this Mac | how many tokens fit in the GPU memory budget after loading the 4-bit weights |
| Fits on this Mac | whether the 4-bit download fits in 90% of the GPU memory budget; models that don't are skipped |

**What "input token cost" means for a local model.** These are open-weight models running on your Mac, so
there is **no per-token fee ($0)**. What each input token actually costs you is:

- **Memory:** every prompt token stays in the KV cache until generation finishes (the KV column).
- **Time:** the prompt must be processed before the first output token appears (measured in section 7).
- **Context:** each token uses up part of the model's fixed context window.
- **Energy:** the GPU draws power while it reads the prompt. Section 7 measures this in millijoules per input
  token and converts it to $ at `ELECTRICITY_USD_PER_KWH`.

Families using grouped-query attention with few KV heads (Qwen2.5, Falcon3) store far less per token than
Phi-3-mini, which has one KV head per attention head.

In [2]:
import mlx.core as mx
import pandas as pd

GB = 1e9
DEVICE = mx.device_info()
GPU_BUDGET_GB = DEVICE['max_recommended_working_set_size'] / GB  # GPU memory macOS allows by default

SPECS = pd.DataFrame([
    # model,                        params,          layers, heads, kv_heads, head_dim, context, download_gb, gated_original
    ('Phi-3-mini-128k-instruct',    3_821_079_552,   32,     32,    32,       96,       131_072, 2.15,        'no'),
    ('Phi-3-small-128k-instruct',   7_392_272_384,   32,     32,    8,        128,      131_072, 4.16,        'no'),
    ('Phi-3-medium-128k-instruct',  13_960_238_080,  40,     40,    10,       128,      131_072, 7.85,        'no'),
    ('Llama-3.2-1B-Instruct',       1_235_814_400,   16,     32,    8,        64,       131_072, 0.70,        'yes'),
    ('Llama-3.2-3B-Instruct',       3_212_749_824,   28,     24,    8,        128,      131_072, 1.81,        'yes'),
    ('Llama-3.1-8B-Instruct',       8_030_261_248,   32,     32,    8,        128,      131_072, 4.52,        'yes'),
    ('Llama-3.3-70B-Instruct',      70_553_706_496,  80,     64,    8,        128,      131_072, 39.69,       'yes'),
    ('Falcon3-1B-Instruct',         1_669_408_768,   18,     8,     4,        256,      8_192,   0.94,        'no'),
    ('Falcon3-3B-Instruct',         3_227_655_168,   22,     12,    4,        256,      32_768,  1.82,        'no'),
    ('Falcon3-7B-Instruct',         7_455_550_464,   28,     12,    4,        256,      32_768,  4.19,        'no'),
    ('Falcon3-10B-Instruct',        10_305_653_760,  40,     12,    4,        256,      32_768,  5.80,        'no'),
    ('Qwen2.5-0.5B-Instruct',       494_032_768,     24,     14,    2,        64,       32_768,  0.28,        'no'),
    ('Qwen2.5-1.5B-Instruct',       1_543_714_304,   28,     12,    2,        128,      32_768,  0.87,        'no'),
    ('Qwen2.5-3B-Instruct',         3_085_938_688,   36,     16,    2,        128,      32_768,  1.74,        'no'),
    ('Qwen2.5-7B-Instruct',         7_615_616_512,   28,     28,    4,        128,      32_768,  4.28,        'no'),
    ('Qwen2.5-14B-Instruct',        14_770_033_664,  48,     40,    8,        128,      32_768,  8.31,        'no'),
    ('Qwen2.5-32B-Instruct',        32_763_876_352,  64,     40,    8,        128,      32_768,  18.43,       'no'),
    ('Qwen2.5-72B-Instruct',        72_706_203_648,  80,     64,    8,        128,      32_768,  40.90,       'no'),
    ('Llama-2-7b-hf',               6_738_415_616,   32,     32,    32,       128,      4_096,   3.79,        'yes'),
    ('Llama-2-13b-hf',              13_015_866_880,  40,     40,    40,       128,      4_096,   7.32,        'yes'),
    ('Llama-2-70b-hf',              68_976_653_312,  80,     64,    8,        128,      4_096,   38.80,       'yes'),
], columns=['model', 'params', 'layers', 'heads', 'kv_heads', 'head_dim', 'context', 'download_gb', 'gated']).set_index('model')

SPECS['kv_bytes_per_token'] = 2 * SPECS.layers * SPECS.kv_heads * SPECS.head_dim * 2
SPECS['q4_gb'] = SPECS.params * 4.5 / 8 / GB
SPECS['full_ctx_gb'] = SPECS.q4_gb + SPECS.kv_bytes_per_token * SPECS.context / GB
SPECS['max_ctx_here'] = ((GPU_BUDGET_GB * 0.9 - SPECS.q4_gb) * GB / SPECS.kv_bytes_per_token).clip(lower=0, upper=SPECS.context).astype(int)
SPECS['fits'] = SPECS.download_gb < GPU_BUDGET_GB * 0.9   # leaves room for the KV cache and macOS

# Output subfolder per model: outputs/ttt/<family>/<size>/<params>/<prompt set>/
def out_dir(name):
    family, size, params = MODELS[name]
    d = Path('outputs/ttt') / family / size / params / PROMPT_SET
    d.mkdir(parents=True, exist_ok=True)
    return d

def check_fits(name):
    if not SPECS.loc[name, 'fits']:
        raise MemoryError(f"{name} needs {SPECS.loc[name, 'download_gb']:.1f} GB for its 4-bit weights, more than this "
                          f"Mac's GPU can use ({GPU_BUDGET_GB:.1f} GB). Pick a smaller model.")

rows = []
for name, (family, size, params) in MODELS.items():
    s = SPECS.loc[name]
    rows.append({
        '': '▶' if name == model_name else '',
        'Family': family,
        'Size': size,
        'Params': params,
        'Model': name,
        'Params (B)': f'{s.params / 1e9:.2f}',
        'Context': f'{s.context:,}',
        'Weights bf16 (GB)': f'{s.params * 2 / GB:.1f}',
        'Weights 8-bit (GB)': f'{s.params * 8.5 / 8 / GB:.1f}',
        'Weights 4-bit (GB)': f'{s.q4_gb:.1f}',
        '4-bit download (GB)': f'{s.download_gb:.2f}',
        'KV cache / input token (KB)': f'{s.kv_bytes_per_token / 1024:.0f}',
        'Memory at full context, 4-bit (GB)': f'{s.full_ctx_gb:.1f}',
        'Max context on this Mac': f'{s.max_ctx_here:,}',
        'API cost / input token': '$0 (local)',
        'Fits on this Mac': 'yes' if s.fits else 'no',
        'Original gated on HF': s.gated,
    })

print(f"{DEVICE['device_name']}: {DEVICE['memory_size'] / GB:.0f} GB unified memory, "
      f"{GPU_BUDGET_GB:.1f} GB usable by the GPU by default.  ▶ = selected model")
pd.DataFrame(rows).style.hide(axis='index')

Apple M5 Max: 52 GB unified memory, 40.2 GB usable by the GPU by default.  ▶ = selected model


,Family,Size,Params,Model,Params (B),Context,Weights bf16 (GB),Weights 8-bit (GB),Weights 4-bit (GB),4-bit download (GB),KV cache / input token (KB),"Memory at full context, 4-bit (GB)",Max context on this Mac,API cost / input token,Fits on this Mac,Original gated on HF
,Phi-3,SMALL,3.8B,Phi-3-mini-128k-instruct,3.82,"131,072",7.6,4.1,2.1,2.15,384,53.7,"86,546",$0 (local),yes,no
,Phi-3,MEDIUM,7B,Phi-3-small-128k-instruct,7.39,"131,072",14.8,7.9,4.2,4.16,128,21.3,"131,072",$0 (local),yes,no
,Phi-3,LARGE,14B,Phi-3-medium-128k-instruct,13.96,"131,072",27.9,14.8,7.9,7.85,200,34.7,"131,072",$0 (local),yes,no
,Llama-3,SMALL,1B,Llama-3.2-1B-Instruct,1.24,"131,072",2.5,1.3,0.7,0.70,32,5.0,"131,072",$0 (local),yes,yes
,Llama-3,SMALL,3B,Llama-3.2-3B-Instruct,3.21,"131,072",6.4,3.4,1.8,1.81,112,16.8,"131,072",$0 (local),yes,yes
▶,Llama-3,MEDIUM,8B,Llama-3.1-8B-Instruct,8.03,"131,072",16.1,8.5,4.5,4.52,128,21.7,"131,072",$0 (local),yes,yes
,Llama-3,LARGE,70B,Llama-3.3-70B-Instruct,70.55,"131,072",141.1,75.0,39.7,39.69,320,82.6,0,$0 (local),no,yes
,Falcon3,SMALL,1B,Falcon3-1B-Instruct,1.67,"8,192",3.3,1.8,0.9,0.94,72,1.5,"8,192",$0 (local),yes,no
,Falcon3,SMALL,3B,Falcon3-3B-Instruct,3.23,"32,768",6.5,3.4,1.8,1.82,88,4.8,"32,768",$0 (local),yes,no
,Falcon3,MEDIUM,7B,Falcon3-7B-Instruct,7.46,"32,768",14.9,7.9,4.2,4.19,112,8.0,"32,768",$0 (local),yes,no


The Llama originals on Hugging Face need an approved access request, but the `mlx-community` 4-bit
versions used here download without one. To try other precisions, swap a repo in `MLX_REPOS` for its
`-8bit` or bf16 variant, or convert an original yourself with
`mlx_lm.convert --hf-path <original repo> --mlx-path <folder> -q --q-bits 8`.

## 3. Load the model

Some models are prepared in `models/` the first time they are selected (see `LOCAL_BUILDS`), then load from disk:

- **Phi-3-small** has no working MLX upload, so it is converted from Microsoft's original checkpoint
  (about 15 GB download) to a 4-bit MLX model.
- **Phi-3-medium**'s MLX upload is fine, but its config uses a legacy name for its RoPE scaling type that
  transformers 5 rejects. A local copy links to the downloaded weights and fixes the name.
- **Llama-2** (7B, 13B, 70B) has no MLX upload, so each is converted from an ungated copy of Meta's checkpoint
  (about 13 GB / 26 GB / 138 GB download).

A model whose 4-bit weights don't fit in this Mac's GPU memory stops here with a clear error.

In [3]:
import json
import shutil
import time
from huggingface_hub import snapshot_download
from mlx.utils import tree_flatten
from mlx_lm import convert, load

def prepare_local_model(name, repo):
    method, src = LOCAL_BUILDS[name]
    dst = Path(repo)
    if method == 'convert':
        print(f'Converting {src} -> {repo} (4-bit), one-time step...')
        src_path = snapshot_download(src, token=False, allow_patterns=[
            '*.json', '*.safetensors', '*.py', '*.model', '*.tiktoken', '*.txt'])
        convert(hf_path=src_path, mlx_path=repo, quantize=True, q_bits=4, q_group_size=64,
                trust_remote_code=name in TRUST_REMOTE_CODE)
    elif method == 'fix_rope':
        print(f'Building {repo} from {src} with the rope type fixed, one-time step...')
        snap = Path(snapshot_download(src))
        dst.mkdir(parents=True)
        for f in snap.iterdir():
            if f.suffix == '.safetensors':
                (dst / f.name).symlink_to(f.resolve())  # link to the Hugging Face cache, no copy
            elif f.name != 'config.json':
                shutil.copy(f, dst / f.name)
        cfg = json.loads((snap / 'config.json').read_text())
        if cfg.get('rope_scaling', {}).get('type') == 'su':
            cfg['rope_scaling']['type'] = 'longrope'
        (dst / 'config.json').write_text(json.dumps(cfg, indent=2))

def load_mlx_model(name):
    repo, _ = MLX_REPOS[name]
    trust = name in TRUST_REMOTE_CODE
    if name in LOCAL_BUILDS and not Path(repo).exists():
        prepare_local_model(name, repo)
    t0 = time.perf_counter()
    model, tokenizer = load(repo, tokenizer_config={'trust_remote_code': True} if trust else None)
    template = Path(repo) / 'chat_template.jinja'
    if tokenizer.chat_template is None and template.exists():
        # Custom tokenizers (Phi-3-small) don't read the template file saved by mlx_lm.convert
        tokenizer._tokenizer.chat_template = template.read_text()
    for token in EXTRA_STOP_TOKENS.get(name, []):
        tokenizer.add_eos_token(token)
    load_s = time.perf_counter() - t0
    n_params = sum(v.size for _, v in tree_flatten(model.parameters()))
    return model, tokenizer, load_s, n_params

check_fits(model_name)
model, tokenizer, load_seconds, n_stored = load_mlx_model(model_name)
spec = SPECS.loc[model_name]
print(f'Loaded {repo} in {load_seconds:.1f}s on {mx.default_device()}')
print(f'Parameters: {spec.params / 1e9:.2f}B  |  layers {spec.layers}, heads {spec.heads}, '
      f'KV heads {spec.kv_heads}, context {spec.context:,}')
print(f'Weights in memory: {mx.get_active_memory() / GB:.2f} GB (4-bit)')

/Users/genailab/mlx-projects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 1783.55it/s]

Loaded mlx-community/Meta-Llama-3.1-8B-Instruct-4bit in 0.9s on Device(gpu, 0)
Parameters: 8.03B  |  layers 32, heads 32, KV heads 8, context 131,072
Weights in memory: 4.52 GB (4-bit)


## 4. Load the prompt set

Every prompt in `INPUT_PATH` is sent to the model as a separate conversation, wrapped in the model's own
chat template. A prompt that asks for a length ("In 500 words") gets that number recorded as its word target,
so section 7 can check how closely each model follows it.

In [4]:
import re

PROMPTS = pd.read_csv(INPUT_PATH, dtype={'id': str})
PROMPTS['id'] = PROMPTS['id'].str.zfill(2)
PROMPTS['words_target'] = PROMPTS.prompt.str.extract(r'[Ii]n (\d+) words', expand=False).astype('Int64')

BASE_STOP = '\n### '   # a base model starting a new section has finished its answer

def build_prompt(tokenizer, text):
    if tokenizer.chat_template is None:
        # Base (non-chat) models such as Llama-2-7b-hf have no chat template, so the request is written as a
        # document that the model continues with the answer
        prompt = f'Below is an instruction, followed by a response that completes it.\n\n### Instruction:\n{text}\n\n### Response:\n'
        return prompt, tokenizer.encode(prompt)
    messages = [{'role': 'user', 'content': text}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return prompt, tokenizer.encode(prompt, add_special_tokens=False)

def encode_all(tokenizer):
    return [build_prompt(tokenizer, text)[1] for text in PROMPTS.prompt]

input_ids_all = encode_all(tokenizer)
n_inputs = [len(ids) for ids in input_ids_all]
print(f'Prompt set: {INPUT_PATH} ({len(PROMPTS)} prompts, {sum(n_inputs):,} input tokens with chat formatting)')
print(f'Largest prompt uses {max(n_inputs) / spec.context:.2%} of the {spec.context:,}-token context window')
pd.DataFrame({
    'id': PROMPTS.id,
    'prompt': PROMPTS.prompt.str.slice(0, 90).where(PROMPTS.prompt.str.len() <= 90, PROMPTS.prompt.str.slice(0, 87) + '...'),
    'input tokens': n_inputs,
    'word target': PROMPTS.words_target,
}).style.hide(axis='index')

Prompt set: prompts/tti_tts_ttt_prompts/ttt_prompts_500w.csv (10 prompts, 606 input tokens with chat formatting)
Largest prompt uses 0.07% of the 131,072-token context window


id,prompt,input tokens,word target
01,Draft a professional email to a manager requesting a recommendation for a new job oppor...,57,500
02,"Provide a brief background on the UK Prime Minister in 1995, highlighting key accomplis...",63,500
03,"Explain the current phase of the moon, its significance, and how it fits within the lun...",61,500
04,"Describe the logic used to calculate the time for baking 96 cookies, considering oven c...",60,500
05,"Imagine three friends, Mia, Noah, and Olivia, each running a race. Mia finishes ahead o...",91,500
06,Tell me about Global warming.(In 500 words),46,500
07,Tell the History of USA.(In 500 words),46,500
08,"Draft a compelling advertisement for a new grocery store in a small town, including the...",69,500
09,"Explain the basic Blackjack game, including rules, logic, and example gameplay.(In 500 ...",56,500
10,Breakdown the steering and breaking components of a Cessna 172N. (In 500 words),57,500


## 5. Generate

Greedy decoding (temperature 0), so results are reproducible.

**Power monitoring.** `macmon` (no sudo needed) runs in the background for the whole notebook, sampling the
chip's power sensors about every 110 ms. Each run's energy is split into the **prefill** window (reading the
prompt, up to the first output token) and the **decode** window (generating). Samples that straddle a window
edge are split by overlap. If `macmon` isn't installed, the power metrics are left empty and everything else runs.

Limits: macmon reports Apple's on-chip power estimates, not a wall-meter reading. Prefill often lasts under half
a second, so only a few samples cover it, and per-input-token energy is approximate.

In [5]:
import atexit
import json
import shutil
import subprocess
import threading

class PowerMonitor:
    """Runs `macmon pipe` in the background and keeps every sample.

    Each sample is the average over the interval since the previous one, so the energy in any time
    window is the sum of power x (overlap of that interval with the window).
    """
    def __init__(self, interval_ms):
        self.samples = []  # (time.perf_counter(), macmon sample dict)
        self.available = shutil.which('macmon') is not None
        if not self.available:
            print('macmon not found: power and energy metrics will be empty (install: brew install macmon)')
            return
        self.proc = subprocess.Popen(['macmon', 'pipe', '-i', str(interval_ms)],
                                     stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        atexit.register(self.proc.terminate)
        threading.Thread(target=self._read, daemon=True).start()
        t_start = time.perf_counter()
        while len(self.samples) < 2 and time.perf_counter() - t_start < 5:  # macmon takes ~0.7 s to start
            time.sleep(0.05)
        self.available = len(self.samples) >= 2

    def _read(self):
        for line in self.proc.stdout:
            try:
                self.samples.append((time.perf_counter(), json.loads(line)))
            except json.JSONDecodeError:
                pass

    def wait_past(self, t, timeout=2.0):
        """Block until a sample covering time t has arrived."""
        deadline = time.perf_counter() + timeout
        while self.samples[-1][0] < t and time.perf_counter() < deadline:
            time.sleep(0.02)

    def window(self, a, b):
        """Samples overlapping [a, b], as (seconds of overlap, sample)."""
        return [(min(t, b) - max(t_prev, a), s)
                for (t_prev, _), (t, s) in zip(self.samples, self.samples[1:])
                if min(t, b) - max(t_prev, a) > 0]

def soc_power(s):
    return s['all_power'] + s['ram_power']  # CPU + GPU + Neural Engine + memory, in watts

def integrate(win, fn):
    return sum(overlap * fn(s) for overlap, s in win)

POWER_KEYS = ['gpu_util_avg_pct', 'gpu_util_peak_pct', 'gpu_freq_avg_mhz', 'gpu_power_avg_w', 'gpu_power_peak_w',
              'soc_power_avg_w', 'gpu_temp_max_c', 'energy_j', 'energy_net_j',
              'prefill_energy_mj_per_input_token', 'decode_energy_mj_per_output_token', 'power_samples']

def summarize_power(t0, t_first, t_end, n_input, n_output):
    if not POWER.available:
        return dict.fromkeys(POWER_KEYS)
    POWER.wait_past(t_end)
    run = POWER.window(t0, t_end)
    samples = [s for _, s in run]
    dur = t_end - t0
    e_soc = integrate(run, soc_power)
    return {
        'gpu_util_avg_pct': round(100 * integrate(run, lambda s: s['gpu_active_ratio']) / dur, 1),
        'gpu_util_peak_pct': round(100 * max(s['gpu_active_ratio'] for s in samples), 1),
        'gpu_freq_avg_mhz': round(integrate(run, lambda s: s['gpu_freq_mhz']) / dur),
        'gpu_power_avg_w': round(integrate(run, lambda s: s['gpu_power']) / dur, 1),
        'gpu_power_peak_w': round(max(s['gpu_power'] for s in samples), 1),
        'soc_power_avg_w': round(e_soc / dur, 1),
        'gpu_temp_max_c': round(max(s['temp']['gpu_temp_avg'] for s in samples), 1),
        'energy_j': round(e_soc, 2),
        'energy_net_j': round(e_soc - IDLE_SOC_W * dur, 2),
        'prefill_energy_mj_per_input_token': round(1000 * integrate(POWER.window(t0, t_first), soc_power) / n_input, 3),
        'decode_energy_mj_per_output_token': round(1000 * integrate(POWER.window(t_first, t_end), soc_power) / max(n_output, 1), 2),
        'power_samples': len(run),
    }

POWER = PowerMonitor(POWER_SAMPLE_MS)
if POWER.available:
    time.sleep(1.0)  # idle baseline, measured with the model loaded but not running
    a, b = POWER.samples[0][0], POWER.samples[-1][0]
    IDLE_SOC_W = integrate(POWER.window(a, b), soc_power) / (b - a)
    print(f'Power monitor running (macmon, ~{POWER_SAMPLE_MS} ms samples). Idle chip power: {IDLE_SOC_W:.2f} W')

def cpu_clusters():
    """CPU cluster names and core counts from macOS (e.g. M5 Max: 6 'Super' + 12 'Performance' cores)."""
    def sysctl(key):
        return subprocess.run(['sysctl', '-n', key], capture_output=True, text=True).stdout.strip()
    return [(sysctl(f'hw.perflevel{i}.name') or f'level{i}', int(sysctl(f'hw.perflevel{i}.logicalcpu') or 0)) for i in (0, 1)]

CPU_CLUSTERS = cpu_clusters()
HARDWARE = {}   # model name -> hardware_metrics() for its whole run

def hardware_metrics(t0, t1):
    """GPU, CPU, Neural Engine (NPU), memory and whole-chip activity over [t0, t1], from the macmon samples.

    macmon calls the two CPU clusters 'pcpu' (perflevel0) and 'ecpu' (perflevel1); they are reported here
    under the names macOS gives them.
    """
    if not POWER.available:
        return None
    POWER.wait_past(t1)
    win = POWER.window(t0, t1)
    samples = [s for _, s in win]
    dur = t1 - t0
    avg = lambda fn: integrate(win, fn) / dur
    peak = lambda fn: max(fn(s) for s in samples)
    energy = lambda fn: round(integrate(win, fn), 2)
    pct = lambda x: round(100 * x, 1)
    (c0_name, c0_cores), (c1_name, c1_cores) = CPU_CLUSTERS
    c0, c1 = c0_name.lower(), c1_name.lower()
    ane = lambda s: s['ane_power']
    return {
        'gpu': {
            'avg_utilization_pct': pct(avg(lambda s: s['gpu_active_ratio'])),
            'peak_utilization_pct': pct(peak(lambda s: s['gpu_active_ratio'])),
            'avg_frequency_mhz': round(avg(lambda s: s['gpu_freq_mhz'])),
            'max_frequency_mhz': round(peak(lambda s: s['gpu_freq_mhz'])),
            'avg_power_watts': round(avg(lambda s: s['gpu_power']), 1),
            'peak_power_watts': round(peak(lambda s: s['gpu_power']), 1),
            'energy_joules': energy(lambda s: s['gpu_power']),
            'avg_temperature_c': round(avg(lambda s: s['temp']['gpu_temp_avg']), 1),
            'max_temperature_c': round(peak(lambda s: s['temp']['gpu_temp_avg']), 1),
        },
        'cpu': {
            'avg_utilization_pct': pct(avg(lambda s: s['cpu_active_ratio'])),
            'peak_utilization_pct': pct(peak(lambda s: s['cpu_active_ratio'])),
            f'{c0}_cores': c0_cores,
            f'{c0}_cores_avg_utilization_pct': pct(avg(lambda s: s['pcpu_active_ratio'])),
            f'{c0}_cores_avg_frequency_mhz': round(avg(lambda s: s['pcpu_freq_mhz'])),
            f'{c1}_cores': c1_cores,
            f'{c1}_cores_avg_utilization_pct': pct(avg(lambda s: s['ecpu_active_ratio'])),
            f'{c1}_cores_avg_frequency_mhz': round(avg(lambda s: s['ecpu_freq_mhz'])),
            'avg_power_watts': round(avg(lambda s: s['cpu_power']), 1),
            'peak_power_watts': round(peak(lambda s: s['cpu_power']), 1),
            'energy_joules': energy(lambda s: s['cpu_power']),
            'avg_temperature_c': round(avg(lambda s: s['temp']['cpu_temp_avg']), 1),
            'max_temperature_c': round(peak(lambda s: s['temp']['cpu_temp_avg']), 1),
        },
        'npu': {
            'name': 'Apple Neural Engine (ANE)',
            'used': peak(ane) > 0.05,
            'avg_power_watts': round(avg(ane), 2),
            'peak_power_watts': round(peak(ane), 2),
            'energy_joules': energy(ane),
            'note': 'MLX runs models on the GPU, so the Neural Engine stays idle (about 0 W) during these runs.',
        },
        'system_memory': {
            'avg_power_watts': round(avg(lambda s: s['ram_power']), 1),
            'energy_joules': energy(lambda s: s['ram_power']),
            'ram_total_gb': round(samples[0]['memory']['ram_total'] / 1e9, 1),
            'avg_ram_used_gb': round(avg(lambda s: s['memory']['ram_usage']) / 1e9, 1),
            'peak_ram_used_gb': round(peak(lambda s: s['memory']['ram_usage']) / 1e9, 1),
            'peak_swap_used_gb': round(peak(lambda s: s['memory']['swap_usage']) / 1e9, 2),
            'note': 'RAM used is for the whole Mac, including other apps, not just this model.',
        },
        'whole_chip': {
            'avg_power_watts': round(avg(soc_power), 1),
            'peak_power_watts': round(peak(soc_power), 1),
            'energy_joules': energy(soc_power),
            'idle_power_watts': round(IDLE_SOC_W, 2),
            'note': 'CPU + GPU + Neural Engine + memory.',
        },
        'measured_seconds': round(dur, 2),
        'power_samples': len(win),
    }

Power monitor running (macmon, ~100 ms samples). Idle chip power: 1.01 W


### Per-prompt metrics analysis

Every prompt is wrapped in `metrics_monitor_start()` / `metrics_monitor_stop()` from `macmon_metrics/metrics_monitor.py`,
which records a macmon time series for that prompt. Results go to a separate folder,
`metrics_analysis/ttt/<family>/<size>/<params>/<prompt set>/`:

- `System_Level_Metrics/<prompt id>_system_metrics.csv`: CPU / GPU / NPU utilisation, power and temperature, RAM used
  and total system power, sampled every 100 ms while that prompt ran
- `Model_Level_Metrics/<model>_prompt_metrics.csv`: one row per prompt with input tokens, output tokens, the first
  token generated and its latency (time to first token), inference rate (tokens/s) and inference time

Section 9 combines every model's `Model_Level_Metrics` CSV into `metrics_analysis/ttt/all_models_prompt_metrics_<prompt set>.csv`.

In [6]:
import macmon_metrics.metrics_monitor as mm
from macmon_metrics.metrics_monitor import metrics_monitor_start, metrics_monitor_stop, setup_output_directory
from mlx_lm import stream_generate
from mlx_lm.sample_utils import make_sampler

METRICS_ROOT = Path('metrics_analysis/ttt')

def analysis_dirs(name):
    """(System_Level_Metrics, Model_Level_Metrics) folders for one model and prompt set."""
    family, size, params = MODELS[name]
    return setup_output_directory(str(METRICS_ROOT / family / size / params / PROMPT_SET))

# Start the macmon tracker now: it takes about a second to produce its first sample, and starting it here means
# every prompt's capture has samples from its first moment.
if mm._tracker is None:
    mm._tracker = mm.MacMonTracker(binary_path=mm._resolve_macmon_binary())
    mm._tracker.start()
    time.sleep(1.5)

def generate(model, tokenizer, input_ids):
    mx.reset_peak_memory()
    text = ''
    t_first = None
    first_token = None
    t0 = time.perf_counter()
    for r in stream_generate(model, tokenizer, input_ids, max_tokens=MAX_NEW_TOKENS, sampler=make_sampler(temp=0.0)):
        if t_first is None:
            t_first = time.perf_counter()  # first token arrives once the prompt has been processed
            first_token = r.text
        text += r.text
        if tokenizer.chat_template is None and BASE_STOP in text.lstrip():
            text = text.lstrip().split(BASE_STOP)[0]
            break
    t_end = time.perf_counter()
    return {
        'text': text,
        'total_s': t_end - t0,
        'output_tokens': r.generation_tokens,
        'prefill_s': r.prompt_tokens / r.prompt_tps,
        'decode_tps': r.generation_tps,
        'peak_gb': r.peak_memory,
        'finish_reason': r.finish_reason or 'stop',   # None when stopped early at BASE_STOP
        'ttft_s': t_first - t0,
        'first_token': first_token,
        'power': summarize_power(t0, t_first, t_end, len(input_ids), r.generation_tokens),
    }

def warm_up(model, tokenizer):
    """One short generation so GPU kernels are compiled before anything is timed."""
    for _ in stream_generate(model, tokenizer, tokenizer.encode('Hello'), max_tokens=8):
        pass

def save_prompt_metrics(name, runs, folder):
    """Model_Level_Metrics CSV: one row per prompt."""
    family, size, _ = MODELS[name]
    rows = [{
        'prompt_id': g['id'],
        'model': name,
        'family': family,
        'size': size,
        'input_tokens': g['input_tokens'],
        'output_tokens': g['output_tokens'],
        'output_words': g['words'],
        'first_token': json.dumps(g['first_token'])[1:-1],   # escaped, so newlines/spaces stay visible in the CSV
        'time_to_first_token (ms)': round(1000 * g['ttft_s'], 1),
        'prompt_processing_rate (tokens/s)': round(g['input_tokens'] / g['prefill_s'], 1),
        'inference_rate (tokens/s)': round(g['decode_tps'], 2),
        'inference_time (s)': round(g['total_s'], 3),
        'prompt': g['prompt'],
        'response': g['text'],
    } for g in runs]
    path = Path(folder) / f'{name}_prompt_metrics.csv'
    pd.DataFrame(rows).to_csv(path, index=False)
    return path

def run_prompt_set(model, tokenizer, name):
    warm_up(model, tokenizer)
    sys_dir, model_dir = analysis_dirs(name)
    runs = []
    t_start = time.perf_counter()
    for row, ids in zip(PROMPTS.itertuples(index=False), encode_all(tokenizer)):
        metrics_monitor_start(idx=row.id, metrics_folder=sys_dir)
        is_plugged = mm._tracker.capture_is_plugged
        g = generate(model, tokenizer, ids)
        metrics_monitor_stop()
        g['is_plugged'] = is_plugged
        g['text'] = g['text'].strip()
        g.update(id=row.id, prompt=row.prompt, input_tokens=len(ids), words=len(g['text'].split()),
                 words_target=None if pd.isna(row.words_target) else int(row.words_target))
        runs.append(g)
        print(f"  [{row.id}] {len(ids):>5} in -> {g['output_tokens']:>4} out, {g['words']:>4} words, "
              f"{g['decode_tps']:5.1f} tok/s, {g['total_s']:5.1f}s ({g['finish_reason']})")
    HARDWARE[name] = hardware_metrics(t_start, time.perf_counter())
    print(f'Per-prompt metrics: {save_prompt_metrics(name, runs, model_dir)} (+ {len(runs)} system CSVs in {sys_dir})')
    return runs

print(f'Running {len(PROMPTS)} prompts on {model_name}...')
runs = run_prompt_set(model, tokenizer, model_name)
print(f"Done: {sum(g['output_tokens'] for g in runs):,} tokens in {sum(g['total_s'] for g in runs):.1f}s")

Running 10 prompts on Llama-3.1-8B-Instruct...


  [01]    57 in ->  508 out,  438 words, 114.9 tok/s,   4.5s (stop)


  [02]    63 in ->  586 out,  427 words, 114.7 tok/s,   5.2s (stop)


  [03]    61 in ->  573 out,  457 words, 114.7 tok/s,   5.1s (stop)


  [04]    60 in ->  725 out,  588 words, 114.4 tok/s,   6.5s (stop)


...

  [06]    46 in ->  712 out,  525 words, 114.5 tok/s,   6.3s (stop)


  [07]    46 in ->  816 out,  571 words, 114.2 tok/s,   7.3s (stop)


  [08]    69 in ->  580 out,  403 words, 114.7 tok/s,   5.2s (stop)


  [09]    56 in ->  750 out,  527 words, 114.3 tok/s,   6.7s (stop)


  [10]    57 in ->  646 out,  503 words, 114.5 tok/s,   5.8s (stop)
Per-prompt metrics: metrics_analysis/ttt/Llama-3/MEDIUM/8B/ttt_prompts_500w/Model_Level_Metrics/Llama-3.1-8B-Instruct_prompt_metrics.csv (+ 10 system CSVs in metrics_analysis/ttt/Llama-3/MEDIUM/8B/ttt_prompts_500w/System_Level_Metrics)
Done: 6,423 tokens in 57.3s


## 6. Save the responses

One text file per prompt: `outputs/ttt/<family>/<size>/<params>/<prompt set>/<model>_response_<id>.txt`.

In [7]:
import textwrap

def save_responses(name, runs):
    d = out_dir(name)
    for g in runs:
        (d / f"{name}_response_{g['id']}.txt").write_text(g['text'] + '\n')
    return d

saved_to = save_responses(model_name, runs)
print(f'Saved {len(runs)} responses to {saved_to}/\n')
first = runs[0]
print(f"=== Prompt {first['id']}: {textwrap.shorten(first['prompt'], 200)}\n")
print(textwrap.fill(first['text'][:1500], width=100, replace_whitespace=False) + (' ...' if len(first['text']) > 1500 else ''))

Saved 10 responses to outputs/ttt/Llama-3/MEDIUM/8B/ttt_prompts_500w/

=== Prompt 01: Draft a professional email to a manager requesting a recommendation for a new job opportunity. (In 500 words)

Subject: Request for Professional Recommendation for New Job Opportunity

Dear [Manager's Name],

I
hope this email finds you well. As you are aware, I have been with the company for [X] years and
have had the privilege of working under your guidance and mentorship. I am writing to request a
professional recommendation for a new job opportunity that I am considering.

As you know, I have
been an integral part of the [Department/Team] and have made significant contributions to the
company's growth and success. I believe that my skills, experience, and achievements make me an
attractive candidate for this new role, and I would be grateful if you could provide a
recommendation to support my application.

The new job opportunity is at [Company Name], where I
would be working as a [Job Title] in t

## 7. Cost of this run

The input-token costs from section 2 and the energy it took, measured per prompt and totalled over the set.
Electricity cost = energy per token × `ELECTRICITY_USD_PER_KWH` (1 kWh = 3.6 MJ).
**Hit limit** means the response was cut off at `MAX_NEW_TOKENS`.

In [8]:
def usd_per_1m(mj_per_token):
    """Electricity cost of one million tokens: mJ/token -> kWh (1 kWh = 3.6e9 mJ) -> $."""
    if mj_per_token is None:
        return None
    return round(mj_per_token * 1e6 / 3.6e9 * ELECTRICITY_USD_PER_KWH, 5)

def run_metrics(name, n_input, gen, load_s):
    s = SPECS.loc[name]
    kv = int(s.kv_bytes_per_token)
    return {
        'family': MODELS[name][0],
        'model': name,
        'repo': MLX_REPOS[name][0],
        'params_b': round(s.params / 1e9, 2),
        'load_s': round(load_s, 1),
        'input_tokens': n_input,
        'output_tokens': gen['output_tokens'],
        'prefill_s': round(gen['prefill_s'], 2),
        'prefill_ms_per_input_token': round(gen['prefill_s'] * 1000 / n_input, 3),
        'prefill_tok_per_s': round(n_input / gen['prefill_s']),
        'decode_tok_per_s': round(gen['decode_tps'], 1),
        'kv_kb_per_token': round(kv / 1024),
        'kv_cache_mb_this_prompt': round(kv * (n_input + gen['output_tokens']) / 1e6, 1),
        'context_used_pct': round(100 * (n_input + gen['output_tokens']) / s.context, 2),
        'peak_memory_gb': round(gen['peak_gb'], 2),
        'api_cost_usd': 0.0,
        **gen['power'],
        'usd_per_1m_input_tokens': usd_per_1m(gen['power']['prefill_energy_mj_per_input_token']),
        'usd_per_1m_output_tokens': usd_per_1m(gen['power']['decode_energy_mj_per_output_token']),
        'electricity_usd_per_kwh': ELECTRICITY_USD_PER_KWH,
    }

def weighted(per, key, weight):
    """Average of per-prompt values weighted by another column (None if power wasn't measured)."""
    pairs = [(p[key], p[weight]) for p in per if p[key] is not None]
    total = sum(w for _, w in pairs)
    return round(sum(v * w for v, w in pairs) / total, 3) if pairs and total else None

def set_metrics(name, runs, load_s):
    per = []
    for g in runs:
        m = run_metrics(name, g['input_tokens'], g, load_s)
        per.append({'id': g['id'], 'words': g['words'], 'words_target': g['words_target'],
                    'finish_reason': g['finish_reason'], 'total_s': round(g['total_s'], 2),
                    **{k: v for k, v in m.items() if k not in
                       ('family', 'model', 'repo', 'params_b', 'load_s', 'api_cost_usd', 'electricity_usd_per_kwh')}})
    s = SPECS.loc[name]
    tot_in = sum(p['input_tokens'] for p in per)
    tot_out = sum(p['output_tokens'] for p in per)
    prefill_s = sum(g['prefill_s'] for g in runs)
    decode_s = sum(g['output_tokens'] / g['decode_tps'] for g in runs)
    measured = per[0]['energy_j'] is not None
    targets = [p for p in per if p['words_target']]
    summary = {
        'family': MODELS[name][0],
        'model': name,
        'repo': MLX_REPOS[name][0],
        'params_b': round(s.params / 1e9, 2),
        'prompt_set': PROMPT_SET,
        'prompts': len(per),
        'load_s': round(load_s, 1),
        'total_input_tokens': tot_in,
        'total_output_tokens': tot_out,
        'total_time_s': round(sum(p['total_s'] for p in per), 1),
        'avg_words': round(sum(p['words'] for p in per) / len(per)),
        'avg_words_vs_target_pct': round(100 * sum(p['words'] / p['words_target'] for p in targets) / len(targets)) if targets else None,
        'prompts_hit_token_limit': sum(p['finish_reason'] == 'length' for p in per),
        'prefill_tok_per_s': round(tot_in / prefill_s),
        'decode_tok_per_s': round(tot_out / decode_s, 1),
        'peak_memory_gb': max(p['peak_memory_gb'] for p in per),
        'kv_kb_per_token': per[0]['kv_kb_per_token'],
        'api_cost_usd': 0.0,
        'gpu_util_avg_pct': weighted(per, 'gpu_util_avg_pct', 'total_s'),
        'gpu_power_avg_w': weighted(per, 'gpu_power_avg_w', 'total_s'),
        'gpu_power_peak_w': max(p['gpu_power_peak_w'] for p in per) if measured else None,
        'soc_power_avg_w': weighted(per, 'soc_power_avg_w', 'total_s'),
        'gpu_temp_max_c': max(p['gpu_temp_max_c'] for p in per) if measured else None,
        'energy_j': round(sum(p['energy_j'] for p in per), 1) if measured else None,
        'energy_net_j': round(sum(p['energy_net_j'] for p in per), 1) if measured else None,
        'prefill_energy_mj_per_input_token': weighted(per, 'prefill_energy_mj_per_input_token', 'input_tokens'),
        'decode_energy_mj_per_output_token': weighted(per, 'decode_energy_mj_per_output_token', 'output_tokens'),
        'electricity_usd_per_kwh': ELECTRICITY_USD_PER_KWH,
    }
    summary['usd_per_1m_input_tokens'] = usd_per_1m(summary['prefill_energy_mj_per_input_token'])
    summary['usd_per_1m_output_tokens'] = usd_per_1m(summary['decode_energy_mj_per_output_token'])
    return {'summary': summary, 'per_prompt': per}

result = set_metrics(model_name, runs, load_seconds)
sm = result['summary']

rows = [
    ('Prompts', f"{sm['prompts']} ({sm['prompts_hit_token_limit']} hit the {MAX_NEW_TOKENS}-token limit)"),
    ('Input / output tokens (total)', f"{sm['total_input_tokens']:,} / {sm['total_output_tokens']:,}"),
    ('Total generation time', f"{sm['total_time_s']} s"),
    ('Average response length', f"{sm['avg_words']} words" + (f" ({sm['avg_words_vs_target_pct']}% of target)" if sm['avg_words_vs_target_pct'] else '')),
    ('Prefill speed / decode speed', f"{sm['prefill_tok_per_s']:,} / {sm['decode_tok_per_s']} tok/s"),
    ('Memory per token (KV cache)', f"{sm['kv_kb_per_token']} KB"),
    ('Peak memory (weights + cache)', f"{sm['peak_memory_gb']} GB"),
    ('API cost', '$0.00 (runs locally)'),
]
if sm['energy_j'] is not None:
    rows += [
        ('GPU utilisation (avg) / GPU power (avg, peak)', f"{sm['gpu_util_avg_pct']:.1f}% / {sm['gpu_power_avg_w']:.1f} W, {sm['gpu_power_peak_w']} W"),
        ('Chip power, avg (CPU + GPU + memory)', f"{sm['soc_power_avg_w']:.1f} W"),
        ('GPU temperature (max)', f"{sm['gpu_temp_max_c']} °C"),
        ('Energy for the whole set', f"{sm['energy_j']} J ({sm['energy_net_j']} J above idle)"),
        ('Energy per input / output token', f"{sm['prefill_energy_mj_per_input_token']:.1f} / {sm['decode_energy_mj_per_output_token']:.1f} mJ"),
        (f'Electricity per 1M input / output tokens (at ${ELECTRICITY_USD_PER_KWH}/kWh)', f"${sm['usd_per_1m_input_tokens']} / ${sm['usd_per_1m_output_tokens']}"),
    ]
display(pd.DataFrame(rows, columns=['Metric', model_name]).style.hide(axis='index'))

pd.DataFrame([{
    'id': p['id'], 'input tokens': p['input_tokens'], 'output tokens': p['output_tokens'],
    'words': p['words'], 'target': p['words_target'] or '', 'hit limit': 'yes' if p['finish_reason'] == 'length' else '',
    'time (s)': f"{p['total_s']:.1f}", 'decode tok/s': f"{p['decode_tok_per_s']:.1f}",
    'energy (J)': '' if p['energy_j'] is None else f"{p['energy_j']:.1f}",
} for p in result['per_prompt']]).style.hide(axis='index')

Metric,Llama-3.1-8B-Instruct
Prompts,10 (0 hit the 1024-token limit)
Input / output tokens (total),"606 / 6,423"
Total generation time,57.3 s
Average response length,488 words (98% of target)
Prefill speed / decode speed,801 / 114.5 tok/s
Memory per token (KV cache),128 KB
Peak memory (weights + cache),4.76 GB
API cost,$0.00 (runs locally)
"GPU utilisation (avg) / GPU power (avg, peak)","98.8% / 23.0 W, 41.4 W"
"Chip power, avg (CPU + GPU + memory)",41.1 W


id,input tokens,output tokens,words,target,hit limit,time (s),decode tok/s,energy (J)
01,57,508,438,500,,4.5,114.9,185.2
02,63,586,427,500,,5.2,114.7,214.7
03,61,573,457,500,,5.1,114.7,209.5
04,60,725,588,500,,6.5,114.4,268.5
05,91,527,445,500,,4.7,114.7,194.9
06,46,712,525,500,,6.3,114.5,262.6
07,46,816,571,500,,7.3,114.2,300.2
08,69,580,403,500,,5.2,114.7,211.3
09,56,750,527,500,,6.7,114.3,274.1
10,57,646,503,500,,5.8,114.5,235.1


### Simple metrics

The metrics saved for each model, as `<model>_simple_metrics.json` in its output folder: plain names with the unit in every key, grouped into tokens, response length, speed,
memory, energy and cost, plus one short entry per prompt. The hardware blocks cover the whole run: **gpu**, **cpu** (per core cluster), **npu** (Apple Neural Engine), **system_memory** and **whole_chip**.

In [9]:
def simple_metrics(name, result):
    """A flat, readable summary of one model's run: plain names, units in every key."""
    sm, per = result['summary'], result['per_prompt']
    s = SPECS.loc[name]
    _, size, params = MODELS[name]
    n = len(per)
    words = [p['words'] for p in per]
    targets = [p['words_target'] for p in per if p['words_target']]
    gen_s = sum(p['total_s'] for p in per)
    energy = sm['energy_j']
    measured = energy is not None
    price = sm['electricity_usd_per_kwh']
    r1 = lambda x: None if x is None else round(x, 1)
    return {
        'model': name,
        'family': sm['family'],
        'size': size,
        'parameters': params,
        'repo': sm['repo'],
        'parameters_billion': sm['params_b'],
        'quantization': '4-bit',
        'context_window_tokens': int(s.context),
        'prompt_set': sm['prompt_set'],
        'prompts_run': n,
        'tokens': {
            'input_total': sm['total_input_tokens'],
            'output_total': sm['total_output_tokens'],
            'input_avg_per_prompt': round(sm['total_input_tokens'] / n),
            'output_avg_per_prompt': round(sm['total_output_tokens'] / n),
            'output_limit_per_prompt': MAX_NEW_TOKENS,
        },
        'response_length': {
            'avg_words': round(sum(words) / n),
            'min_words': min(words),
            'max_words': max(words),
            'target_words': max(set(targets), key=targets.count) if targets else None,
            'avg_pct_of_target': sm['avg_words_vs_target_pct'],
            'prompts_within_10pct_of_target': sum(1 for p in per if p['words_target'] and abs(p['words'] / p['words_target'] - 1) <= 0.1),
            'prompts_cut_off_at_token_limit': sm['prompts_hit_token_limit'],
        },
        'speed': {
            'model_load_seconds': sm['load_s'],
            'total_generation_seconds': round(gen_s, 1),
            'avg_seconds_per_prompt': round(gen_s / n, 2),
            'avg_time_to_first_token_ms': round(1000 * sum(p['prefill_s'] for p in per) / n),
            'prompt_reading_tokens_per_second': sm['prefill_tok_per_s'],
            'generation_tokens_per_second': sm['decode_tok_per_s'],
            'ms_per_output_token': round(1000 / sm['decode_tok_per_s'], 2),
            'words_per_second': round(sum(words) / gen_s, 1),
        },
        'memory': {
            'weights_gb': float(s.download_gb),
            'peak_memory_gb': sm['peak_memory_gb'],
            'kv_cache_kb_per_token': sm['kv_kb_per_token'],
            'kv_cache_mb_largest_request': max(p['kv_cache_mb_this_prompt'] for p in per),
            'context_window_used_max_pct': max(p['context_used_pct'] for p in per),
        },
        **(HARDWARE.get(name) or {'hardware': 'not measured (macmon not available)'}),
        'energy': {
            'total_joules': energy,
            'total_watt_hours': round(energy / 3600, 3) if measured else None,
            'avg_joules_per_prompt': round(energy / n, 1) if measured else None,
            'per_input_token_millijoules': sm['prefill_energy_mj_per_input_token'],
            'per_output_token_millijoules': sm['decode_energy_mj_per_output_token'],
            'per_1000_words_joules': round(1000 * energy / sum(words), 1) if measured else None,
        },
        'cost_usd': {
            'api_fee': 0.0,
            'electricity_whole_run': round(energy / 3.6e6 * price, 6) if measured else None,
            'electricity_per_1m_input_tokens': sm['usd_per_1m_input_tokens'],
            'electricity_per_1m_output_tokens': sm['usd_per_1m_output_tokens'],
            'electricity_price_per_kwh': price,
        },
        'per_prompt': [{
            'id': p['id'],
            'words': p['words'],
            'target_words': p['words_target'],
            'input_tokens': p['input_tokens'],
            'output_tokens': p['output_tokens'],
            'seconds': p['total_s'],
            'time_to_first_token_ms': round(1000 * p['prefill_s']),
            'tokens_per_second': p['decode_tok_per_s'],
            'energy_joules': p['energy_j'],
            'cut_off_at_token_limit': p['finish_reason'] == 'length',
        } for p in per],
    }

def save_simple(name, result, folder):
    path = Path(folder) / f'{name}_simple_metrics.json'
    path.write_text(json.dumps(simple_metrics(name, result), indent=2))
    return path

# --- write it for the model run above
print(f'Saved {save_simple(model_name, result, out_dir(model_name))}')
print(json.dumps({k: v for k, v in simple_metrics(model_name, result).items() if k != 'per_prompt'}, indent=2))

Saved outputs/ttt/Llama-3/MEDIUM/8B/ttt_prompts_500w/Llama-3.1-8B-Instruct_simple_metrics.json
{
  "model": "Llama-3.1-8B-Instruct",
  "family": "Llama-3",
  "size": "MEDIUM",
  "parameters": "8B",
  "repo": "mlx-community/Meta-Llama-3.1-8B-Instruct-4bit",
  "parameters_billion": 8.03,
  "quantization": "4-bit",
  "context_window_tokens": 131072,
  "prompt_set": "ttt_prompts_500w",
  "prompts_run": 10,
  "tokens": {
    "input_total": 606,
    "output_total": 6423,
    "input_avg_per_prompt": 61,
    "output_avg_per_prompt": 642,
    "output_limit_per_prompt": 1024
  },
  "response_length": {
    "avg_words": 488,
    "min_words": 403,
    "max_words": 588,
    "target_words": 500,
    "avg_pct_of_target": 98,
    "prompts_within_10pct_of_target": 4,
    "prompts_cut_off_at_token_limit": 0
  },
  "speed": {
    "model_load_seconds": 0.9,
    "total_generation_seconds": 57.3,
    "avg_seconds_per_prompt": 5.73,
    "avg_time_to_first_token_ms": 76,
    "prompt_reading_tokens_per_second"

## 8. Compare every model in the family

Runs the whole prompt set through every model of `FAMILY` (all SMALL, MEDIUM and LARGE parameter counts),
reusing the results already produced for the selected model and unloading each model before the next.
Models too big for this Mac's GPU memory are skipped. Set `TTT_COMPARE=0` (or `RUN_FAMILY_COMPARISON = False`
in section 1) to skip this section.

In [10]:
import gc

label = lambda name: f'{MODELS[name][1]} {MODELS[name][2]}'   # e.g. 'SMALL 1.5B'
results = {label(model_name): (result, runs)}
if RUN_FAMILY_COMPARISON:
    del model, tokenizer
    gc.collect()
    mx.clear_cache()
    for name, (family, size, params) in MODELS.items():
        if family != FAMILY or name == model_name:
            continue
        if not SPECS.loc[name, 'fits']:
            print(f"{size} {params}: {name} skipped, needs {SPECS.loc[name, 'download_gb']:.1f} GB (GPU budget {GPU_BUDGET_GB:.1f} GB)")
            continue
        print(f'{size} {params}: {name}')
        m, tok, load_s, _ = load_mlx_model(name)
        r = run_prompt_set(m, tok, name)
        save_responses(name, r)
        res = set_metrics(name, r, load_s)
        save_simple(name, res, out_dir(name))
        results[label(name)] = (res, r)
        del m, tok
        gc.collect()
        mx.clear_cache()

order = [label(n) for n, (f, _, _) in MODELS.items() if f == FAMILY and label(n) in results]
compare = pd.DataFrame({
    s: {
        'Model': sm_['model'],
        'Params (B)': f"{sm_['params_b']:.2f}",
        'Total input / output tokens': f"{sm_['total_input_tokens']:,} / {sm_['total_output_tokens']:,}",
        'Avg words (target %)': f"{sm_['avg_words']}" + (f" ({sm_['avg_words_vs_target_pct']}%)" if sm_['avg_words_vs_target_pct'] else ''),
        'Prompts hitting token limit': sm_['prompts_hit_token_limit'],
        'Total time (s)': sm_['total_time_s'],
        'Prefill (tok/s)': f"{sm_['prefill_tok_per_s']:,}",
        'Decode (tok/s)': sm_['decode_tok_per_s'],
        'Peak memory (GB)': sm_['peak_memory_gb'],
        'GPU util avg (%)': sm_['gpu_util_avg_pct'],
        'GPU power avg (W)': sm_['gpu_power_avg_w'],
        'Energy, whole set (J)': sm_['energy_j'],
        'Energy / input token (mJ)': sm_['prefill_energy_mj_per_input_token'],
        'Energy / output token (mJ)': sm_['decode_energy_mj_per_output_token'],
        '$ / 1M output tokens (electricity)': sm_['usd_per_1m_output_tokens'],
    } for s in order for sm_ in [results[s][0]['summary']]
})
compare.columns.name = FAMILY
compare

SMALL 1B: Llama-3.2-1B-Instruct

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 3176.70it/s]

  [01]    57 in ->  428 out,  373 words, 525.1 tok/s,   0.9s (stop)


  [02]    63 in ->  818 out,  660 words, 523.3 tok/s,   1.6s (stop)


...

  [07]    46 in ->  805 out,  521 words, 237.9 tok/s,   3.5s (stop)


  [08]    69 in ->  535 out,  377 words, 239.9 tok/s,   2.3s (stop)


  [09]    56 in ->  749 out,  539 words, 238.1 tok/s,   3.2s (stop)


  [10]    57 in ->  688 out,  538 words, 238.7 tok/s,   3.0s (stop)
Per-prompt metrics: metrics_analysis/ttt/Llama-3/SMALL/3B/ttt_prompts_500w/Model_Level_Metrics/Llama-3.2-3B-Instruct_prompt_metrics.csv (+ 10 system CSVs in metrics_analysis/ttt/Llama-3/SMALL/3B/ttt_prompts_500w/System_Level_Metrics)
LARGE 70B: Llama-3.3-70B-Instruct skipped, needs 39.7 GB (GPU budget 40.2 GB)


Llama-3,SMALL 1B,SMALL 3B,MEDIUM 8B
Model,Llama-3.2-1B-Instruct,Llama-3.2-3B-Instruct,Llama-3.1-8B-Instruct
Params (B),1.24,3.21,8.03
Total input / output tokens,"606 / 7,286","606 / 6,103","606 / 6,423"
Avg words (target %),533 (107%),456 (91%),488 (98%)
Prompts hitting token limit,1,0,0
Total time (s),14.4,26.4,57.3
Prefill (tok/s),"2,204","1,504",801
Decode (tok/s),531.3,239.0,114.5
Peak memory (GB),0.84,2.07,4.76
GPU util avg (%),94.38,97.077,98.76


In [11]:
# Words per response, per prompt and size
pd.DataFrame({s: {r['id']: r['words'] for r in results[s][1]} for s in order}).rename_axis('prompt id')

,SMALL 1B,SMALL 3B,MEDIUM 8B
prompt id,,,
01,373,362,438
02,660,413,427
03,567,429,457
04,498,368,588
05,486,506,445
06,575,511,525
07,688,521,571
08,408,377,403
09,476,539,527


In [12]:
# The first prompt's response from each size, side by side
for s in order:
    r = results[s][1][0]
    print(f"=== {s}: {results[s][0]['summary']['model']} ({r['words']} words) ===")
    print(textwrap.fill(r['text'][:800], width=100, replace_whitespace=False) + (' ...' if len(r['text']) > 800 else ''), end='\n\n')

=== SMALL 1B: Llama-3.2-1B-Instruct (373 words) ===
Subject: Request for Professional Recommendation

Dear [Manager's Name],

I hope this email finds
you well. I am writing to request a professional recommendation for a new job opportunity that I
have been considering. As you may know, I have been with [Company Name] for [Number] years and have
consistently demonstrated my value to the organization through my contributions to the team and my
commitment to delivering high-quality results.

Recently, I have been presented with an exceptional
opportunity to take on new challenges and responsibilities, which aligns with my career goals and
aspirations. I am excited about the prospect of joining a new team and contributing to the
organization's success. However, I am also aware that this new opportunity may require me to take on
new responsibili ...

=== SMALL 3B: Llama-3.2-3B-Instruct (362 words) ===
Subject: Request for Professional Recommendation for Upcoming Job Opportunity

Dear [Manag

## 9. All models on this prompt set

Collects every `<model>_simple_metrics.json` produced so far for `PROMPT_SET` (any family, any size, from this
notebook or terminal runs) into `outputs/ttt/all_models_<prompt set>.json` and `.csv`.

In [13]:
FAMILY_ORDER = list(MODEL_NAME[PLATFORM][TASK])
SIZE_ORDER = ['SMALL', 'MEDIUM', 'LARGE']
files = Path('outputs/ttt').glob(f'*/*/*/{PROMPT_SET}/*_simple_metrics.json')
models = sorted((json.loads(f.read_text()) for f in files),
                key=lambda m: (FAMILY_ORDER.index(m['family']), SIZE_ORDER.index(m['size']), m['parameters_billion']))
overview = [{k: v for k, v in m.items() if k != 'per_prompt'} for m in models]
all_json = Path('outputs/ttt') / f'all_models_{PROMPT_SET}.json'
all_csv = all_json.with_suffix('.csv')
all_json.write_text(json.dumps({'prompt_set': PROMPT_SET, 'models': overview}, indent=2))
flat = pd.json_normalize(overview)
flat.to_csv(all_csv, index=False)
print(f'{len(models)} models -> {all_json} and {all_csv}')

pd.DataFrame({
    'Family': flat['family'], 'Size': flat['size'], 'Params': flat['parameters'], 'Model': flat['model'],
    'Params (B)': flat['parameters_billion'],
    'Avg words': flat['response_length.avg_words'],
    'Cut off': flat['response_length.prompts_cut_off_at_token_limit'],
    'Gen tok/s': flat['speed.generation_tokens_per_second'],
    'First token (ms)': flat['speed.avg_time_to_first_token_ms'],
    'Total time (s)': flat['speed.total_generation_seconds'],
    'Peak memory (GB)': flat['memory.peak_memory_gb'],
    'GPU busy (%)': flat.get('gpu.avg_utilization_pct'),
    'GPU power (W)': flat.get('gpu.avg_power_watts'),
    'CPU busy (%)': flat.get('cpu.avg_utilization_pct'),
    'CPU power (W)': flat.get('cpu.avg_power_watts'),
    'NPU power (W)': flat.get('npu.avg_power_watts'),
    'Energy (J)': flat['energy.total_joules'],
    'mJ / output token': flat['energy.per_output_token_millijoules'],
    '$ / 1M output tokens': flat['cost_usd.electricity_per_1m_output_tokens'],
}).style.hide(axis='index')

13 models -> outputs/ttt/all_models_ttt_prompts_500w.json and outputs/ttt/all_models_ttt_prompts_500w.csv


Family,Size,Params,Model,Params (B),Avg words,Cut off,Gen tok/s,First token (ms),Total time (s),Peak memory (GB),GPU busy (%),GPU power (W),CPU busy (%),CPU power (W),NPU power (W),Energy (J),mJ / output token,$ / 1M output tokens
Phi-3,SMALL,3.8B,Phi-3-mini-128k-instruct,3.820000,684,9,180.500000,46,56.200000,2.730000,98.600000,23.600000,7.300000,5.200000,0.000000,2297.800000,228.429000,0.009520
Phi-3,MEDIUM,7B,Phi-3-small-128k-instruct,7.390000,346,0,102.800000,70,42.700000,4.400000,97.900000,23.900000,6.200000,4.800000,0.000000,1753.800000,403.931000,0.016830
Phi-3,LARGE,14B,Phi-3-medium-128k-instruct,13.960000,667,6,61.800000,106,159.000000,8.200000,99.400000,18.900000,5.900000,4.000000,0.000000,5861.300000,598.558000,0.024940
Llama-3,SMALL,1B,Llama-3.2-1B-Instruct,1.240000,533,1,531.300000,28,14.400000,0.840000,91.200000,21.700000,10.100000,7.100000,0.000000,578.300000,78.597000,0.003270
Llama-3,SMALL,3B,Llama-3.2-3B-Instruct,3.210000,456,0,239.000000,40,26.400000,2.070000,95.400000,34.600000,6.800000,5.700000,0.000000,1405.300000,228.480000,0.009520
Llama-3,MEDIUM,8B,Llama-3.1-8B-Instruct,8.030000,488,0,114.500000,76,57.300000,4.760000,98.200000,22.800000,4.600000,4.600000,0.000000,2356.200000,362.304000,0.015100
Falcon3,SMALL,3B,Falcon3-3B-Instruct,3.230000,496,1,246.000000,38,28.800000,2.050000,95.600000,27.300000,7.800000,4.900000,0.000000,1272.000000,183.960000,0.007660
Falcon3,MEDIUM,7B,Falcon3-7B-Instruct,7.460000,500,0,115.200000,66,60.500000,4.380000,98.300000,23.200000,5.800000,4.200000,0.000000,2449.900000,355.365000,0.014810
Falcon3,LARGE,10B,Falcon3-10B-Instruct,10.310000,450,0,81.600000,92,77.700000,6.040000,98.300000,23.100000,7.000000,4.300000,0.000000,3149.400000,500.697000,0.020860
Qwen2.5,SMALL,3B,Qwen2.5-3B-Instruct,3.090000,552,1,215.200000,47,35.300000,1.960000,96.900000,30.900000,8.800000,6.800000,0.000000,1725.100000,231.462000,0.009640


In [14]:
# Every model's per-prompt metrics (Model_Level_Metrics) in one CSV
prompt_csvs = sorted(Path('metrics_analysis/ttt').glob(f'*/*/*/{PROMPT_SET}/Model_Level_Metrics/*_prompt_metrics.csv'))
all_prompts = pd.concat([pd.read_csv(p, dtype={'prompt_id': str}) for p in prompt_csvs], ignore_index=True)
all_prompts['family'] = pd.Categorical(all_prompts['family'], FAMILY_ORDER, ordered=True)
all_prompts['size'] = pd.Categorical(all_prompts['size'], SIZE_ORDER, ordered=True)
all_prompts['params_b'] = all_prompts['model'].map(SPECS.params) / 1e9
all_prompts = all_prompts.sort_values(['family', 'size', 'params_b', 'prompt_id']).drop(columns='params_b')
combined_csv = Path('metrics_analysis/ttt') / f'all_models_prompt_metrics_{PROMPT_SET}.csv'
all_prompts.to_csv(combined_csv, index=False)
print(f'{len(prompt_csvs)} models, {len(all_prompts)} prompt rows -> {combined_csv}')
all_prompts.groupby(['family', 'size', 'model'], observed=True, sort=False).agg(
    prompts=('prompt_id', 'count'), input_tokens=('input_tokens', 'sum'), output_tokens=('output_tokens', 'sum'),
    avg_ttft_ms=('time_to_first_token (ms)', 'mean'), avg_inference_rate=('inference_rate (tokens/s)', 'mean'),
    total_inference_time_s=('inference_time (s)', 'sum')).round(1)

13 models, 130 prompt rows -> metrics_analysis/ttt/all_models_prompt_metrics_ttt_prompts_500w.csv


prompts  input_tokens  \
family  size   model                                               
Phi-3   SMALL  Phi-3-mini-128k-instruct         10           351   
        MEDIUM Phi-3-small-128k-instruct        10           326   
        LARGE  Phi-3-medium-128k-instruct       10           351   
Llama-3 SMALL  Llama-3.2-1B-Instruct            10           606   
               Llama-3.2-3B-Instruct            10           606   
        MEDIUM Llama-3.1-8B-Instruct            10           606   
Falcon3 SMALL  Falcon3-3B-Instruct              10           391   
        MEDIUM Falcon3-7B-Instruct              10           391   
        LARGE  Falcon3-10B-Instruct             10           391   
Qwen2.5 SMALL  Qwen2.5-3B-Instruct              10           571   
        MEDIUM Qwen2.5-7B-Instruct              10           571   
               Qwen2.5-14B-Instruct             10           571   
Llama-2 SMALL  Llama-2-7b-hf                    10           625   

                                           output_tokens  avg_ttft_ms  \
family  size   model                                                    
Phi-3   SMALL  Phi-3-mini-128k-instruct            10027         57.5   
        MEDIUM Phi-3-small-128k-instruct            4306         71.0   
        LARGE  Phi-3-medium-128k-instruct           9743        119.2   
Llama-3 SMALL  Llama-3.2-1B-Instruct                7286         68.2   
               Llama-3.2-3B-Instruct                6103         79.5   
        MEDIUM Llama-3.1-8B-Instruct                6423        114.7   
Falcon3 SMALL  Falcon3-3B-Instruct                  6877         75.1   
        MEDIUM Falcon3-7B-Instruct                  6838        105.3   
        LARGE  Falcon3-10B-Instruct                 6228        133.9   
Qwen2.5 SMALL  Qwen2.5-3B-Instruct                  7374         96.9   
        MEDIUM Qwen2.5-7B-Instruct                  7292        122.0   
               Qwen2.5-14B-Instruct                 6908        158.1   
Llama-2 SMALL  Llama-2-7b-hf                        4720         83.7   

                                           avg_inference_rate  \
family  size   model                                            
Phi-3   SMALL  Phi-3-mini-128k-instruct                 180.6   
        MEDIUM Phi-3-small-128k-instruct                102.9   
        LARGE  Phi-3-medium-128k-instruct                61.8   
Llama-3 SMALL  Llama-3.2-1B-Instruct                    531.8   
               Llama-3.2-3B-Instruct                    239.1   
        MEDIUM Llama-3.1-8B-Instruct                    114.6   
Falcon3 SMALL  Falcon3-3B-Instruct                      247.7   
        MEDIUM Falcon3-7B-Instruct                      115.3   
        LARGE  Falcon3-10B-Instruct                      81.9   
Qwen2.5 SMALL  Qwen2.5-3B-Instruct                      216.2   
        MEDIUM Qwen2.5-7B-Instruct                      121.5   
               Qwen2.5-14B-Instruct                      61.0   
Llama-2 SMALL  Llama-2-7b-hf                            121.0   

                                           total_inference_time_s  
family  size   model                                               
Phi-3   SMALL  Phi-3-mini-128k-instruct                      56.2  
        MEDIUM Phi-3-small-128k-instruct                     42.7  
        LARGE  Phi-3-medium-128k-instruct                   159.0  
Llama-3 SMALL  Llama-3.2-1B-Instruct                         14.4  
               Llama-3.2-3B-Instruct                         26.4  
        MEDIUM Llama-3.1-8B-Instruct                         57.3  
Falcon3 SMALL  Falcon3-3B-Instruct                           28.8  
        MEDIUM Falcon3-7B-Instruct                           60.5  
        LARGE  Falcon3-10B-Instruct                          77.7  
Qwen2.5 SMALL  Qwen2.5-3B-Instruct                           35.3  
        MEDIUM Qwen2.5-7B-Instruct                           61.5  
               Qwen2.5-14B-Instruct                         115.0 